# Prepare the research dataset

**OPTIONS LAB / DATA QUALITY**

Normalize source schemas, validate quotes, and keep the latest observation per contract per UTC date. Source CSVs remain unchanged.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "options_lab").is_dir() else Path.cwd().parent
assert (ROOT / "options_lab").is_dir(), "Run from the repository root or notebooks directory."
sys.path.insert(0, str(ROOT))
from options_lab.data import load_snapshots, make_labels, FEATURES
from options_lab.plotting import set_theme, plot_report
set_theme()
INPUTS = sorted([*ROOT.glob("data/raw/*.csv"), *ROOT.glob("data/snapshots/*.csv")])


## Inspect coverage and exclusions

Missing timestamps, expired contracts, nonpositive bids, crossed quotes, and invalid liquidity fields are excluded. Missing spot prices are retained for train-only imputation.

In [ ]:
snapshots, audit = load_snapshots(INPUTS)
import pandas as pd
display(pd.DataFrame([audit]))
display(snapshots[["contractSymbol", "date_collected", "bid", "ask", "mid"]].head())

## Interpretation

Labels measure the next **observed midpoint change for the same option contract** within seven calendar days. They do not measure expiration ITM. Missing future quotes remain unlabeled. Every experiment uses whole-date train/validation/test splits and purges outcomes that reach the next split. Preprocessing fits on training data only. The majority baseline competes with learned models using validation Brier loss; test data is held out.

The included snapshots have no repeat observations for the same contract. An `insufficient_data` report is the expected result until more history is collected. Synthetic fixtures validate the code in tests; their scores are never published as market evidence.
